# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.3 MB/s eta 0:00:00
dependencies ok


In [4]:
import re
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference,helper,numpy_helper,TensorProto

In [5]:
ROOT = Path.cwd()
TASK_CANDIDATES = [
    Path('/kaggle/input/competitions/neurogolf-2026/task383.json'),
    Path('/mnt/data/task383.json'),
    ROOT / 'recovered' / 'task383.json',
]
TASK = next((p for p in TASK_CANDIDATES if p.exists()), None)
assert TASK is not None, f'task383.json not found in {TASK_CANDIDATES}'
OUT = ROOT / "task383_v7_finalist_consensus"
OUT.mkdir(exist_ok=True)
MODEL = OUT / "task383.onnx"


In [6]:
def const(name, value, dtype=np.float32):
    return numpy_helper.from_array(np.asarray(value, dtype=dtype), name=name)

nodes=[]; init=[]
def C(name,value,dtype=np.float32):
    # Materialize constants as graph nodes. This is robust when Kaggle notebooks
    # split/re-execute export cells and avoids dependence on a live initializer list.
    nodes.append(helper.make_node("Constant",[],[name],value=const(name+"_value",value,dtype)))
    return name
def N(op, ins, outs, **kw): nodes.append(helper.make_node(op,ins,outs,**kw))


In [7]:
# Canvas and color evidence.
C("zero",0.0); C("four",4.0)
C("axes_ch",np.array([1],np.int64),np.int64)
C("axes_w",np.array([3],np.int64),np.int64)
C("axes_h",np.array([2],np.int64),np.int64)
C("axes_unsq",np.array([2,3],np.int64),np.int64)
C("depth",np.array(10,np.int64),np.int64)
C("oh_values",np.array([0.,1.],np.float32))
C("ch0_penalty",np.array([[10000.,0,0,0,0,0,0,0,0,0]],np.float32))

N("ReduceSum",["input","axes_ch"],["sum_ch"],keepdims=1)
N("Greater",["sum_ch","zero"],["active"])
N("ReduceMax",["input"],["color_max"],axes=[2,3],keepdims=0)
N("Greater",["color_max","zero"],["present_b"])
N("Cast",["present_b"],["present"],to=TensorProto.FLOAT)

In [8]:
# The official/finalist convention scans 0, frame, then fill. The frame is the
# first nonzero color, so its channel is the lowest present nonzero channel only
# after color permutation? No: derive it spatially from the first nonzero cell.
pos=np.arange(1,901,dtype=np.float32).reshape(1,1,30,30)
C("pos",pos); C("missing_pos",np.array(1000.,np.float32))
N("Greater",["input","zero"],["pixel_present"])
N("Where",["pixel_present","pos","missing_pos"],["pos_or_missing"])
N("ReduceMin",["pos_or_missing"],["first_pos"],axes=[2,3],keepdims=0)
N("Add",["first_pos","ch0_penalty"],["outer_score"])
N("ArgMin",["outer_score"],["outer_idx"],axis=1,keepdims=0)
N("OneHot",["outer_idx","depth","oh_values"],["outer_flat"],axis=-1)
N("Unsqueeze",["outer_flat","axes_unsq"],["outer_oh"])

# Inner is the other present nonzero color.
N("Sub",["present","outer_flat"],["inner_s0"])
N("Sub",["inner_s0","ch0_penalty"],["inner_score"])
N("ArgMax",["inner_score"],["inner_idx"],axis=1,keepdims=0)
N("OneHot",["inner_idx","depth","oh_values"],["inner_flat"],axis=-1)
N("Unsqueeze",["inner_flat","axes_unsq"],["inner_oh"])

N("Mul",["input","outer_oh"],["outer_pixels"])
N("ReduceSum",["outer_pixels","axes_ch"],["outer_mask_f"],keepdims=1)
N("ReduceSum",["outer_mask_f","axes_w"],["row_count"],keepdims=1)
N("ReduceSum",["outer_mask_f","axes_h"],["col_count"],keepdims=1)
for p in ("row","col"):
    N("Greater",[p+"_count","zero"],[p+"_positive"])
    N("Less",[p+"_count","four"],[p+"_small"])
    N("And",[p+"_positive",p+"_small"],[p+"_flag"])
N("Or",["row_flag","col_flag"],["stripe"])

In [9]:
# On a selected row/column: zero-valued cells -> inner; nonzero -> outer.
C("nonzero_channels",np.array([0,1,1,1,1,1,1,1,1,1],np.float32).reshape(1,10,1,1))
N("Mul",["input","nonzero_channels"],["nz_parts"])
N("ReduceSum",["nz_parts","axes_ch"],["nz_sum"],keepdims=1)
N("Greater",["nz_sum","zero"],["is_nonzero"])
N("Where",["is_nonzero","outer_oh","inner_oh"],["stripe_value"])
N("Where",["stripe","stripe_value","input"],["canvas_output"])
C("zeros",np.zeros((1,10,30,30),np.float32))
N("Where",["active","canvas_output","zeros"],["output"])

In [10]:
graph=helper.make_graph(nodes,"task383_finalist_consensus_v8",
    [helper.make_tensor_value_info("input",TensorProto.FLOAT,[1,10,30,30])],
    [helper.make_tensor_value_info("output",TensorProto.FLOAT,[1,10,30,30])],init)
model=helper.make_model(graph,opset_imports=[helper.make_opsetid("",17)],producer_name="task383-v8")
model.ir_version=8
onnx.checker.check_model(model); onnx.save(model,MODEL)

In [11]:
def pad(grid):
    g=np.asarray(grid,np.int64); h,w=g.shape
    x=np.zeros((1,10,30,30),np.float32)
    rr,cc=np.indices((h,w)); x[0,g,rr,cc]=1
    return x

data=json.loads(TASK.read_text())
sess=ort.InferenceSession(str(MODEL),providers=["CPUExecutionProvider"])

In [12]:
def run(g): return sess.run(None,{"input":pad(g)})[0]
results={}
for split,cases in data.items():
    ok=sum(np.array_equal(run(e["input"]),pad(e["output"])) for e in cases)
    results[split]={"ok":ok,"total":len(cases)}
    if ok != len(cases):
        e=cases[0]; h,w=np.asarray(e['output']).shape
        print('DEBUG',split,run(e['input']).argmax(1)[0,:h,:w])
        print('EXPECT',np.asarray(e['output']))
    assert ok==len(cases),(split,ok,len(cases))

In [13]:
# Published first-, second-, and Sakana-solver consensus.
src1="p=lambda g:[g:=[[[c,Q[2+c%~c]][f'{Q}'[1:8]in'%s'%r]for c in r]for*r,in zip(*g[::-1])]for*Q,in[{}.fromkeys(sum(g,[]))]*4][3]"
src2="p=lambda g:[[(a:=[v,*{}.fromkeys(sum(g,r))])[any(0<i.count(a[2])<4for i in[r,c])*2+0**v]for*c,v in zip(*g,r)]for r in g]"
src3="def p(g):A,F,E,*G,H,G=filter(any,g);B=max(A);C=A.index(B);D=E[C+2];return[[(F,(D,B)[F>0])[D in E[-~C::sum(A)//B-3]+G]for(F,*G)in zip(E,F,H)]for E in g]"
solvers=[]
for src in (src1,src2,src3):
    ns={}; exec(src,ns); solvers.append(ns['p'])
consensus_ok=0
for split,cases in data.items():
    for e in cases:
        outs=[p([r[:] for r in e['input']]) for p in solvers]
        assert outs[0]==outs[1]==outs[2]==e['output']
        h,w=np.asarray(e['output']).shape
        pred=run(e['input']).argmax(1)[0,:h,:w].tolist()
        assert pred==outs[0]
        consensus_ok+=1
results['published_finalist_consensus']={"ok":consensus_ok,"total":consensus_ok}


<string>:1: SyntaxWarning: invalid decimal literal


In [14]:
# Stress the shared policy under rotations, reflections, and arbitrary color relabeling.
rng=random.Random(383)
base_cases=sum((data[k] for k in data),[])
stress_agree=stress_match=0
for _ in range(1200):
    g=np.asarray(rng.choice(base_cases)['input'],dtype=np.int64)
    g=np.rot90(g,rng.randrange(4))
    if rng.randrange(2): g=np.fliplr(g)
    colors=[int(c) for c in np.unique(g) if c]
    remap=dict(zip(colors,rng.sample(range(1,10),len(colors))))
    g=np.vectorize(lambda v:remap.get(int(v),int(v)))(g).tolist()
    outs=[p([r[:] for r in g]) for p in solvers]
    if outs[0]==outs[1]==outs[2]:
        stress_agree+=1
        h,w=len(g),len(g[0])
        stress_match+=run(g).argmax(1)[0,:h,:w].tolist()==outs[0]
assert stress_agree==stress_match==1200
results['transformed_finalist_consensus']={"ok":stress_match,"total":stress_agree}


In [15]:
# Tensor and graph safety checks.
proto=onnx.load(MODEL)
ops=sorted({n.op_type for n in proto.graph.node})
forbidden=sorted(set(ops)&{"Loop","Scan","NonZero","Unique","Script","Function"})
assert not forbidden and not proto.functions and MODEL.stat().st_size<1_400_000
z=sess.run(None,{"input":np.zeros((1,10,30,30),np.float32)})[0]
assert np.count_nonzero(z)==0
summary={
 "task_id":"task383","model_family":"published_finalist_consensus_v8_constant_nodes",
 "results":results,"onnx_size_bytes":MODEL.stat().st_size,
 "onnx_sha256":hashlib.sha256(MODEL.read_bytes()).hexdigest(),
 "node_count":len(proto.graph.node),"ops":ops,"forbidden_ops":forbidden,
 "input_shape":[1,10,30,30],"output_shape":[1,10,30,30],
 "zip_members":["task383.onnx"]
}
(OUT/'task383_v8_validation_summary.json').write_text(json.dumps(summary,indent=2))


for zp in (Path.cwd()/'submission.zip',):
    with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as zf: zf.write(MODEL,'task383.onnx')
    assert zipfile.ZipFile(zp).namelist()==['task383.onnx']
print(json.dumps(summary,indent=2))


{
  "task_id": "task383",
  "model_family": "published_finalist_consensus_v8_constant_nodes",
  "results": {
    "train": {
      "ok": 3,
      "total": 3
    },
    "test": {
      "ok": 1,
      "total": 1
    },
    "arc-gen": {
      "ok": 262,
      "total": 262
    },
    "published_finalist_consensus": {
      "ok": 266,
      "total": 266
    },
    "transformed_finalist_consensus": {
      "ok": 1200,
      "total": 1200
    }
  },
  "onnx_size_bytes": 42316,
  "onnx_sha256": "f58f98b560b3240eef92f5fcd4417cb25d2e0747827f43ce2471f5e178e7aa77",
  "node_count": 47,
  "ops": [
    "Add",
    "And",
    "ArgMax",
    "ArgMin",
    "Cast",
    "Constant",
    "Greater",
    "Less",
    "Mul",
    "OneHot",
    "Or",
    "ReduceMax",
    "ReduceMin",
    "ReduceSum",
    "Sub",
    "Unsqueeze",
    "Where"
  ],
  "forbidden_ops": [],
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "zip_members": [
    "task383.onnx"
  ]
